In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
os.chdir(r"C:\Users\Mohan2\OneDrive - University of Ottawa\THESIS\test_set")

In [4]:
pd_test_set = pd.read_pickle("test_set_final.pkl")
pd_test_set

,fold,domain,PDB_chain,discontinuous,RCO,PDB,ligand,complex,subchains,MSA_mean,MSA_med,MLP,HMM,PLM,PSSM,len,class
0,a1,d4po5e1,4PO5_E,False,0.063099,4PO5,2.0,hetero hexamer,2.0,85.231635,101.922328,0.598765,0.456790,0.907407,0.864198,167,a
1,a2,d1ysma1,1YSM_A,False,0.069242,1YSM,0.0,homo monomer,1.0,14.965283,16.414062,0.854545,0.909091,0.890909,0.854545,77,a
2,a3,d3o5ca2,3O5C_A,False,0.092339,3O5C,3.0,homo dimer,1.0,10403.821582,10808.736136,0.701299,0.587662,0.886364,0.779221,320,a
3,a4,d3e6mb_,3E6M_B,False,0.051875,3E6M,0.0,homo dimer,1.0,84434.482901,83552.761616,0.714286,0.639456,0.904762,0.863946,161,a
4,a5,d2bwba_,2BWB_A,False,0.100927,2BWB,0.0,homo monomer,1.0,314.387269,374.223147,0.826087,0.826087,0.956522,0.869565,46,a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1225,g12,d1n7da7,1N7D_A,False,0.028927,1N7D,2.0,homo monomer,1.0,18264.782769,18117.513894,0.638498,0.455399,0.766823,0.830986,699,g
1226,g28,d1icfi_,1ICF_I,False,0.093218,1ICF,1.0,hetero trimer,3.0,17.470998,18.576909,0.815385,0.584615,0.953846,0.784615,65,g
1227,g55,d1e8pa_,1E8P_A,False,0.201535,1E8P,0.0,homo monomer,1.0,11.275423,13.755937,0.652174,0.521739,0.760870,0.652174,46,g
1228,g60,d1lsla2,1LSL_A,False,0.121650,1LSL,2.0,homo monomer,1.0,4131.257207,4261.632869,0.734513,0.601770,0.884956,0.761062,113,g


In [4]:
MLP_predictions  = np.load("MLP_predictions2.npz")
HMM_predictions  = np.load("HMM_predictions2.npz")
PLM_predictions  = np.load("PLM_predictions2.npz")
PSSM_predictions = np.load("p1217.npz")
all_chains       = np.load("ac2.npz")

In [5]:
def get_accuracy(prediction, ground_truth):
    assert len(prediction)==len(ground_truth), print(len(prediction), len(ground_truth))
    mask = ground_truth!= -9
    prediction   = prediction[mask]
    ground_truth = ground_truth[mask]
    
    return sum(prediction==ground_truth)/len(ground_truth)

In [11]:
pd_test_set['MLP'] = None
pd_test_set['HMM'] = None
pd_test_set['PLM'] = None
pd_test_set['PSSM']= None
pd_test_set['len'] = None

for row in pd_test_set.index:
    PDB_ID = pd_test_set.loc[row]['PDB_chain']
    try:
        ground_truth = all_chains[PDB_ID][1]
        pd_test_set.at[row, 'len'] = len(ground_truth)
        pd_test_set.at[row, 'MLP'] = get_accuracy(MLP_predictions[PDB_ID], ground_truth)
        pd_test_set.at[row, 'HMM'] = get_accuracy(HMM_predictions[PDB_ID], ground_truth)
        pd_test_set.at[row, 'PLM'] = get_accuracy(PLM_predictions[PDB_ID].squeeze(), ground_truth)
        pd_test_set.at[row, 'PSSM']= get_accuracy(PSSM_predictions[PDB_ID], ground_truth)
    except Exception as e:
        print(PDB_ID, e)

1ZT2_B '1ZT2_B is not a file in the archive'
2PSM_C '2PSM_C is not a file in the archive'


In [6]:
import numba as nb

@nb.jit
def _is_equal_nb(seq, subseq, m, i):
    for j in range(m):
        if seq[i + j] != subseq[j]:
            return False
    return True


@nb.jit
def find_loop_nb(seq, subseq):
    n = len(seq)
    m = len(subseq)
    for i in range(n - m + 1):
        if _is_equal_nb(seq, subseq, m, i):
            yield i

In [15]:
AA_dict = {i:j for i,j in zip(['a','r','n','d','c','q','e','g','h','i','l','k','m','f','p','s','t','w','y','v','x'],range(21))}
AA_dict['X'] = -9

astral_file = open("astral-scopedom-seqres-gd-all-2.08-stable.fa", "r")
domain_dict = {}

line = next(astral_file)
dom_name = line.split()[0][1:]
seq = ""
domain_list = list(pd_test_set['domain'])

for line in astral_file:
    if line[0] == ">":
        if dom_name in domain_list:
            try:
                domain_dict[dom_name] = np.array([AA_dict[x] for x in seq], dtype=np.int8)
                seq = ""
                dom_name = line.split()[0][1:]
            except:
                print(dom_name)
        else:
            seq = ""
            dom_name = line.split()[0][1:]
    else:
        seq = seq + line.strip()

astral_file.close()
# domain_dict = np.load("domain_dict.npz")

In [16]:
seq = 'mhpagiiltkvgyytipsmddlakitnekgecivsdftigrkgygsiyfegdvnltnlnlddivhirrkevvvylddnqkppvgeglnrkaevtldgvwptdktsrclikspdrladinyegrleavsrkqgaqfkeyrpetgswvfkvshfXkyglqd'
domain_dict['d1ko6.1'] = np.array([AA_dict[x] for x in seq], dtype=np.int8)

In [17]:
seq ='mpkvkrsrkappdgwelieptldeldqkmreaetephegkrkveslwpifrihhqktryifdlfykrkaisrelyeycikegyadknliakwkkqgyenlcclrciqtrdtnfgtncicrvpksklevgriiecthcgcrgcs'
domain_dict['d6id1n_'] = np.array([AA_dict[x] for x in seq], dtype=np.int8)

In [18]:
seq = 'tinvtgdgnvfkpsaetsstavpslslspgmlnXpggvpwiaigdetsvtspgalrrmtskdipetaiintdnssgavpsesalvpyndeplvvvtehaianftkaemalefnrefldklrvlsvspkysdlltyvdcyvgvsarqalnnfqkqvpvitptrqtmyvdsiqaalkalekweidlrvaqtllptnvpigevscpmqsvvkllddqlpddslirrypkeaavalakrnggiqwmdvsegtvmneavnavaasalapsasappleeksklteqamdlvtaaepeiiaslvpvpapvfaippkpadynvrtlkideatwlrmipktmgtlfqiqvtdntgtnwhfnlrggtrvvnldqiapmrfvldlggksyketswdpngkkvgfivfqskipfelwtaasqigqatvvnyvqlyaedssftaqsiiattslaynyepeqlnktdpemnyyllatfidsaaitptnmtqpdvwdalltmsplsagevtvkgavvsevvpaeligsytpeslnaslpndaarcmidraskiaeaikidddagpdeyspnsvpiqgqlaisqletgygvrifnpkgilskiasramqafigdpstiitqaapvlsdknnwialaqgvktslrtkslsagvktavsklsssesiqnwtqgfldkvsthfpap'
domain_dict['d1jmu.3'] = np.array([AA_dict[x] for x in seq], dtype=np.int8)

In [19]:
for index in pd_test_set[pd_test_set['discontinuous'].isnull()].index:
    domain = pd_test_set.loc[index]['domain']
    if -9 in domain_dict[domain]:
        pd_test_set.at[index, 'discontinuous'] = True
    else:
        pd_test_set.at[index, 'discontinuous'] = False 

In [132]:
def get_subchain_accuracy(chain, subchain, prediction, ground_truth):
    print(len(prediction))
    stride = next(find_loop_nb(chain, subchain))
    prediction = prediction[stride:stride+len(subchain)]
    ground_truth = ground_truth[stride:stride+len(subchain)]
    mask = ground_truth!=-9
    prediction = prediction[mask]
    ground_truth = ground_truth[mask]
    print(len(prediction))
    accuracy = sum(prediction==ground_truth)/len(prediction)
    
    return(accuracy)

In [129]:
pd_test_set['MLP_domain'] = None
pd_test_set['HMM_domain'] = None
pd_test_set['PLM_domain'] = None
pd_test_set['PSSM_domain']= None
pd_test_set['domain_len'] = None

for row in pd_test_set[(~pd_test_set['discontinuous']) & (~pd_test_set['PDB_chain'].isin(remaining))].index:
    PDB_ID = pd_test_set.loc[row, 'PDB_chain']
    domain = pd_test_set.loc[row, 'domain']
    try:
        main_chain = all_chains[PDB_ID][0]
        sub_seq    = domain_dict[domain]
        ground_truth = all_chains[PDB_ID][1]
        pd_test_set.at[row,'domain_len'] = len(sub_seq)
        if len(main_chain) == len(sub_seq):
            pd_test_set.at[row, 'HMM_domain']  = pd_test_set.loc[row, 'HMM']
            pd_test_set.at[row, 'MLP_domain']  = pd_test_set.loc[row, 'MLP']
            pd_test_set.at[row, 'PSSM_domain'] = pd_test_set.loc[row, 'PSSM']
            pd_test_set.at[row, 'PLM_domain']  = pd_test_set.loc[row, 'PLM']
        else:
            prediction = HMM_predictions[PDB_ID]
            accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
            pd_test_set.at[row, 'HMM_domain'] = accuracy

            prediction = MLP_predictions[PDB_ID]
            accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
            pd_test_set.at[row, 'MLP_domain'] = accuracy

            prediction = PLM_predictions[PDB_ID]
            accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
            pd_test_set.at[row, 'PLM_domain'] = accuracy

            prediction = PSSM_predictions[PDB_ID]
            accuracy = get_subchain_accuracy(main_chain, sub_seq, prediction, ground_truth)
            pd_test_set.at[row, 'PSSM_domain'] = accuracy
        
    except Exception as e:
        print(PDB_ID , e)
        pass

1G3P_A 
1VQ8_O 
1P49_A 
4H27_A 
6MU0_A 
4NDJ_A 
1VQO_X 
2ZCF_A 
4KO2_L 
6KAC_Z 
6LY5_A 
3C75_L 
3TKK_A 


In [114]:
def get_discontinuous_accuracy(chain, subchain, prediction, ground_truth):
    split = np.where(subchain==-9)
    split = split[0]
    subchains = np.split(subchain, split)
    subchains[1] = subchains[1][1:]
    acc1 = get_subchain_accuracy(chain, subchains[0], prediction, ground_truth) * len(subchains[0])
    acc2 = get_subchain_accuracy(chain, subchains[1], prediction, ground_truth) * len(subchains[1])
    return (acc1+acc2)/(len(subchains[0])+len(subchains[1]))

In [130]:
#get domain accuracy for discontinuous domains
for ind in pd_test_set[pd_test_set['discontinuous']].index:
    PDB_ID = pd_test_set.loc[ind, 'PDB_chain']
    domain = pd_test_set.loc[ind, 'domain']
    try:
        main_chain = all_chains[PDB_ID][0]
        sub_seq    = domain_dict[domain]
        ground_truth = all_chains[PDB_ID][1]
        pd_test_set.at[ind, 'domain_len'] = len(sub_seq)
        
        prediction = HMM_predictions[PDB_ID]
        accuracy = get_discontinuous_accuracy(main_chain, sub_seq, prediction, ground_truth)
        pd_test_set.at[ind, 'HMM_domain'] = accuracy

        prediction = MLP_predictions[PDB_ID]
        accuracy = get_discontinuous_accuracy(main_chain, sub_seq, prediction, ground_truth)
        pd_test_set.at[ind, 'MLP_domain'] = accuracy
        
        prediction = PLM_predictions[PDB_ID]
        accuracy = get_discontinuous_accuracy(main_chain, sub_seq, prediction, ground_truth)
        pd_test_set.at[ind, 'PLM_domain'] = accuracy
        
        prediction = PSSM_predictions[PDB_ID]
        accuracy = get_discontinuous_accuracy(main_chain, sub_seq, prediction, ground_truth)
        pd_test_set.at[ind, 'PSSM_domain'] = accuracy

    except Exception as e:
        print(PDB_ID, e)

4G24_A '4G24_A is not a file in the archive'


In [8]:
pd_test_set.to_csv("test_set.csv")

In [101]:
# def find_loop1(seq, subseq):
#     n = len(seq)
#     m = len(subseq)
#     for i in range(n - m + 1):
#         if (sum(seq[i:i+m]==subseq)==m-1):
#             yield i
            
# def get_subchain2(chain, subchain, prediction, ground_truth):
#     stride = next(find_loop1(chain, subchain))
#     prediction = prediction[stride:stride+len(subchain)]
#     ground_truth = ground_truth[stride:stride+len(subchain)]
#     mask = ground_truth!=-9
#     prediction = prediction[mask]
#     ground_truth = ground_truth[mask]

#     accuracy = sum(prediction==ground_truth)/len(prediction)
#     return(accuracy)

# indices = pd_test_set[pd_test_set['PDB_chain'].isin(exceptions)].index
# for row in indices:
#     domain = pd_test_set.loc[row, 'domain']
#     print(domain)
#     sub_seq= domain_dict[domain]
#     main_chain= all_chains[chain][0]
#     ground_truth=all_chains[chain][1]
    
#     prediction = HMM_predictions[chain]
#     accuracy = get_subchain2(main_chain, sub_seq, prediction, ground_truth)
#     pd_test_set.at[row, 'HMM_domain'] = float(accuracy)

#     prediction = MLP_predictions[chain]
#     accuracy = get_subchain2(main_chain, sub_seq, prediction, ground_truth)
#     pd_test_set.at[row, 'MLP_domain'] = accuracy

#     prediction = PLM_predictions[chain]
#     accuracy = get_subchain2(main_chain, sub_seq, prediction, ground_truth)
#     pd_test_set.at[row, 'PLM_domain'] = accuracy

#     prediction = PSSM_predictions[chain]
#     accuracy = get_subchain2(main_chain, sub_seq, prediction, ground_truth)
#     pd_test_set.at[row, 'PSSM_domain'] = accuracy    